In [1]:
import os, json

In [2]:
import time

In [3]:
from together import Together
import utils

In [ ]:
from importlib import reload
reload(utils)

In [4]:
key_file = 'together-lab-key.txt'
with open(key_file, 'r') as f:
    API_KEY = f.read().strip()

client = Together(
  api_key=API_KEY
)


In [6]:
model_name = 'qwq'
model_endpoint = utils.model_names_to_endpoints[model_name]
model_endpoint

'Qwen/QwQ-32B'

basic mc questions

In [7]:
data_dir = '../data/final_dataset'
mc_files = ['certamen_mc.json', 'nle_other_questions_2015.json', 'nle_other_questions_2020.json', 'nle_other_questions_2025.json']
mc_files = [os.path.join(data_dir, f) for f in mc_files]

file_to_data = {}
for file in mc_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

In [5]:
def construct_mc_user_prompt(q_dict):
    question_text = q_dict['question'] if 'question' in q_dict else q_dict['question_text']
    choices = q_dict['multiple_choice_options']
    choices_text = '\n'.join(choices)
    question_text += '\n' + choices_text
    question_text += '\n' + utils.mc_format_instructions

    #if model_name == 'qwq' and thinking:
    #    question_text += '\n<think>\n'
    return question_text


In [9]:
prompt = construct_mc_user_prompt(file_to_data['certamen_mc.json'][0])
prompt

'What Latin preposition is the root of “country”?\nA: CONTRA\nB: CUM\nC: ULTRA\nAt the end of your response, give the letter of the correct answer as\nAnswer: Letter'

In [ ]:
file_to_data['certamen_mc.json'][0]

{'source_name': 'NJCL-Certamen',
 'source_year': 1996,
 'question_id': 'NJCL-Certamen_1996_9a_1',
 'question_format': 'multiple_choice',
 'question_content': 'vocabulary',
 'difficulty': 'unknown',
 'question_language': 'english',
 'answer_language': 'latin',
 'question': 'What Latin preposition is the root of “country”?',
 'multiple_choice_options': ['A: CONTRA', 'B: CUM', 'C: ULTRA'],
 'answers': ['A: CONTRA']}

In [11]:
'''
response = client.chat.completions.create(
  model=model_endpoint,
  messages=[
    {
        "role": "system",
        "content": utils.sys_prompt
    },
    {
      "role": "user",
      "content": prompt
    }
  ],
  temperature=0.6, 
  top_p=0.95, 
  #min_p=0,
  #top_k=20
)
print(response.choices[0].message.content)
'''

'\nresponse = client.chat.completions.create(\n  model=model_endpoint,\n  messages=[\n    {\n        "role": "system",\n        "content": utils.sys_prompt\n    },\n    {\n      "role": "user",\n      "content": prompt\n    }\n  ],\n  temperature=0.6, \n  top_p=0.95, \n  #min_p=0,\n  #top_k=20\n)\nprint(response.choices[0].message.content)\n'

In [12]:
#resp = response.choices[0].message.content
#len(resp.split())

In [13]:
save_dir = f'../data/model_responses/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)


In [14]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0
    save_file = os.path.join(save_dir, filename)
    if os.path.exists(save_file):
        with open(save_file, 'r') as f:
            q_id_to_resp = json.load(f)

    for q_dict in data:
        q_id = q_dict['question_id']
        if q_id in q_id_to_resp:
            i += 1
            continue
        prompt = construct_mc_user_prompt(q_dict)

        response = client.chat.completions.create(
            model=model_endpoint,
            messages=[
                {
                    "role": "system",
                    "content": utils.sys_prompt
                },
                {
                "role": "user",
                "content": prompt
                }
            ],
            temperature=0.6, 
            top_p=0.95, 
            #min_p=0,
            #top_k=20
        )
        try:
            resp = response.choices[0].message.content
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.01)
        

        if i % 10 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)
        


certamen_mc.json
nle_other_questions_2015.json
nle_other_questions_2020.json
  10 / 173
  20 / 173
  30 / 173
  40 / 173
  50 / 173
  60 / 173
  70 / 173
  80 / 173
  90 / 173
  100 / 173
  110 / 173
  120 / 173
  130 / 173
  140 / 173
  150 / 173
  160 / 173
  170 / 173
nle_other_questions_2025.json
  0 / 130
  10 / 130
  20 / 130
  30 / 130
  40 / 130
  50 / 130
  60 / 130
  70 / 130
  80 / 130
  90 / 130
  100 / 130
  110 / 130
  120 / 130


reading comp questions

In [15]:
import glob

In [16]:
key_file = 'together-lab-key.txt'
#key_file = 'together-personal.txt'
with open(key_file, 'r') as f:
    API_KEY = f.read().strip()

client = Together(
  api_key=API_KEY
)


In [17]:
files = glob.glob(os.path.join(data_dir, '*.json'))
question_files = [f for f in files if 'reading_comp_questions' in f]
passage_files = [f for f in files if 'reading_comp_passages' in f]

file_to_data = {}
for file in question_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)
    print(base_name, len(file_to_data[base_name]))

nle_reading_comp_questions_2015.json 103
nle_reading_comp_questions_2020.json 103
nle_reading_comp_questions_2025.json 176


In [18]:
question_files

['../data/final_dataset/nle_reading_comp_questions_2015.json',
 '../data/final_dataset/nle_reading_comp_questions_2020.json',
 '../data/final_dataset/nle_reading_comp_questions_2025.json']

In [19]:
# create passage id to passage text dict
passage_id_to_txt = {}
for passage_file in passage_files:
    with open(passage_file, 'r') as f:
        passage_data = json.load(f)

    for p_dict in passage_data:
        id_ = p_dict['passage_id']
        text = p_dict['text']
        passage_id_to_txt[id_] = text

In [6]:
def construct_rc_user_prompt(q_dict):

    passage_id = q_dict['passage_id']
    passage_text = passage_id_to_txt[passage_id]

    question_text = f'{passage_text}\n\n'

    question_text += q_dict['question'] if 'question' in q_dict else q_dict['question_text']
    choices = q_dict['multiple_choice_options']
    choices_text = '\n'.join(choices)
    question_text += '\n' + choices_text
    question_text += '\n' + utils.mc_format_instructions

    #if model_name == 'qwq' and thinking:
    #    question_text += '\n<think>\n'
    return question_text

In [28]:
prompt = construct_rc_user_prompt(file_to_data['nle_reading_comp_questions_2015.json'][0])
prompt

'READ THE REST OF THE STORY AND ANSWER THE QUESTIONS.\n\nTHE STRUGGLE\nVir Germānicus ex forō fugit. Senātor et duo fīliī virum agitant. Senātor virum comprehendit. Senātor cum virō pugnat. Turba pugnam videt et circumvenit. Vir turbam timet. Vir effugere temptat et inter duōs puerōs currit. Vir forte puerōs offendit et in terram dēcidit.\n"Tū fīliōs meōs offendere audēs," senātor clāmat. "Ego tibi supplicium postulō quod fīliōs meōs vulnerās."\n"Pater," ūnus fīlius inquit, "vir Germānicus forte nōs vulnerābat. Nōlī pūnīre virum. Vir est viātor. Potest portāre litterās ad Germāniam."\n"Ita vērō," senātor respondet, "Tū es callidus."\n\n1 fugit = flees\n2 agitant = chase; comprehendit = takes hold of\n3 Turba = A crowd\n4 circumvenit = surrounds; effugere = to escape\n5 currit = runs; forte = accidentally\n6 offendit = bumps into; dēcidit = falls down\n7 audēs = dare\n8 supplicium postulō = ask for the death penalty; vulnerās = you are hurting\n9\n10 Nōlī pūnīre = Don\'t punish; viātor 

In [29]:
'''
response = client.chat.completions.create(
  model=model_endpoint,
  messages=[
    {
        "role": "system",
        "content": utils.sys_prompt
    },
    {
      "role": "user",
      "content": prompt
    }
  ],
  temperature=0.6, 
  top_p=0.95, 
  #min_p=0,
  #top_k=20
)
print(response.choices[0].message.content)
'''

To determine who is chasing the German man, we look at the text: "Vir Germānicus ex forō fugit. Senātor et duo fīliī virum agitant." This translates to "The German man flees from the forum. The senator and his two sons chase the man."

Given this information, the correct answer is the one that identifies the senator and his two sons as the ones chasing the German man.

Answer: C


In [21]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0
    save_file = os.path.join(save_dir, filename)
    if os.path.exists(save_file):
        with open(save_file, 'r') as f:
            q_id_to_resp = json.load(f)
    for q_dict in data:
        q_id = q_dict['question_id']
        if q_id in q_id_to_resp:
            i += 1
            continue
        prompt = construct_rc_user_prompt(q_dict)

        response = client.chat.completions.create(
            model=model_endpoint,
            messages=[
                {
                    "role": "system",
                    "content": utils.sys_prompt
                },
                {
                "role": "user",
                "content": prompt
                }
            ],
            temperature=0.6, 
            top_p=0.95, 
            #min_p=0,
            #top_k=20
        )
        try:
            resp = response.choices[0].message.content
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.01)
        

        if i % 50 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)
        


nle_reading_comp_questions_2015.json
  0 / 103
  50 / 103
  100 / 103
nle_reading_comp_questions_2020.json
  0 / 103
  50 / 103
  100 / 103
nle_reading_comp_questions_2025.json
  0 / 176
  50 / 176
  100 / 176
  150 / 176


rerun questions

In [16]:
import os, json

In [29]:
model_name = 'llama3-turbo'
model_endpoint = utils.model_names_to_endpoints[model_name]

In [14]:
data_dir = '../data/final_dataset'

In [8]:
#file_basename = 'nle_reading_comp_questions_2015.json'
file_basename = 'certamen_mc.json'
passage_file = 'nle_reading_comp_passages_2015.json'

q_id = "NLE_2015_LATIN_V-VI_30"

In [18]:
passage_data

[{'source_name': 'NLE',
  'source_year': 2015,
  'question_id': 'NLE_2015_INTRODUCTION_TO_LATIN_31',
  'question_format': 'multiple_choice',
  'question_content': 'reading_comprehension',
  'passage_id': 'NLE_2015_INTRODUCTION_TO_LATIN_0',
  'difficulty': 'beginner',
  'question_language': 'english',
  'answer_language': 'english',
  'question': 'In lines 1-2, the German man is being chased by',
  'multiple_choice_options': ['A: two other German men',
   'B: the crowd',
   'C: the senator and his two sons',
   'D: the guards and soldiers'],
  'answers': ['C: the senator and his two sons'],
  'correctness_logic': 'na',
  'n_required': None},
 {'source_name': 'NLE',
  'source_year': 2015,
  'question_id': 'NLE_2015_INTRODUCTION_TO_LATIN_32',
  'question_format': 'multiple_choice',
  'question_content': 'reading_comprehension',
  'passage_id': 'NLE_2015_INTRODUCTION_TO_LATIN_0',
  'difficulty': 'beginner',
  'question_language': 'english',
  'answer_language': 'english',
  'question': 'In

In [20]:
with open(os.path.join(data_dir, file_basename), 'r') as f:
    question_data = json.load(f)
with open(os.path.join(data_dir, passage_file), 'r') as f:
    passage_data = json.load(f)

passage_id_to_txt = {}
for p_dict in passage_data:
    id_ = p_dict['passage_id']
    text = p_dict['text']
    passage_id_to_txt[id_] = text

In [30]:
q_dict = {}
for q in question_data:
    if q['question_id'] == q_id:
        q_dict = q
        break
q_dict

{'source_name': 'NLE',
 'source_year': 2015,
 'question_id': 'NLE_2015_LATIN_V-VI_30',
 'question_format': 'multiple_choice',
 'question_content': 'reading_comprehension',
 'passage_id': 'NLE_2015_LATIN_V-VI_1',
 'difficulty': 'advanced',
 'question_language': 'english',
 'answer_language': 'latin',
 'question': 'The scansion of line 8, a pentameter line of elegiac couplet, is',
 'multiple_choice_options': ['A: - υ υ / - - / - // - υ υ / - υ υ / -',
  'B: - - / - - / - // - υ υ / - υ υ / -',
  'C: - υ υ / - υ υ / - // - υ υ / - υ υ / -',
  'D: - - / - υ υ / - // - υ υ / - υ υ / -'],
 'answers': ['C: - υ υ / - υ υ / - // - υ υ / - υ υ / -'],
 'correctness_logic': 'na',
 'n_required': None}

In [31]:
prompt = construct_rc_user_prompt(q_dict)
prompt

'TERMINUS, THE GOD OF BOUNDARIES AND DOMAIN\nOvid recognizes Terminus\' place and role in Roman society.\n\nQuid, nova cum fierent Capitōlia? Nempe deōrum\ncūncta Iovī cessit turba locumque dedit;\nTerminus, ut veterēs memorant, inventus in aede\nrestitit et magnō cum Iove templa tenet.\nNunc quoque, sẽ suprā nē quid nisi sīdera cernat,\nexiguum templī tēcta foramen habent.\nTermine, post illud levitās tibi lībera nõn est:\nquā positus fuerīs in statiōne, manē;\nnec tū vīcīnō quicquam concēde rogantī,\nnē videāre hominem praeposuisse Iovī:\net seu võmeribus seu tū pulsābere rastrīs,\nclāmātō "tuus est hic ager, ille tuus"...\nGentibus est aliīs tellūs data līmite certō:\nRōmānae spatium est Urbis et orbis idem.\nOvid\'s Fasti, II.667-678, 683-684\n\n1 Quid = Quid accidit; Nempe = Of course\n2\n3\n4\n5 nisi = except\n6 exiguum forāmen = a small opening\n7 illud refers to building the temple; levitās = mobility\n8\n9 vīcīnō = neighboring tribe; quicquam = quidquam\n10\n11 võmeribus = plo

In [32]:
response = client.chat.completions.create(
  model=model_endpoint,
  messages=[
    {
        "role": "system",
        "content": utils.sys_prompt
    },
    {
      "role": "user",
      "content": prompt
    }
  ],
  temperature=0.6, 
  top_p=0.95, 
  #min_p=0,
  #top_k=20
)
resp = response.choices[0].message.content
resp

'To determine the correct scansion of line 8, "Termine, post illud levitās tibi lībera nõn est:", let\'s break it down:\n\n1. "Termine" - This is a dactyl (- υ υ), but since it\'s the first word, it sets the meter for the line. However, in scansion, the first foot can sometimes be a bit flexible, but ideally, it should fit the pattern of the line.\n2. "post illud" - This is a dactyl followed by a spondee (- υ υ / - -).\n3. "levitās" - This is a dactyl (- υ υ).\n4. "tibi" - This is a trochee (- υ), but in the context of elegiac couplets, it often functions as part of a larger metrical foot.\n5. "lībera" - This is a dactyl (- υ υ).\n6. "nõn est" - This is a spondee followed by a trochee, but in the context, it can be seen as part of the final foot, which in elegiac couplets, often ends in a trochee or can be considered as part of the final dactyl (- υ υ).\n\nGiven the pattern of an elegiac couplet, which is typically:\n- υ υ / - - / - // - υ υ / - υ υ / -\n\nThe scansion of "Termine, pos

In [33]:
# replace raw model response
resp_dir = f'../data/model_responses/{model_name}'

resp_file = os.path.join(resp_dir, file_basename)
resp_file

'../data/model_responses/llama3-turbo/nle_reading_comp_questions_2015.json'

In [34]:
with open(resp_file, 'r') as f:
    resp_data = json.load(f)
resp_data

{'NLE_2015_INTRODUCTION_TO_LATIN_31': 'To determine who is chasing the German man, we need to look at lines 1-2 of the story. The relevant sentence is: "Senātor et duo fīliī virum agitant." This translates to "The senator and his two sons chase the man." \n\nTherefore, the German man is being chased by the senator and his two sons.\n\nAnswer: C',
 'NLE_2015_INTRODUCTION_TO_LATIN_32': 'To determine who caught the man, we need to look at the Latin text in line 2: "Senātor virum comprehendit." Here, "Senātor" means senator, "virum" means the man, and "comprehendit" means takes hold of or catches. Therefore, it is the senator who catches the man.\n\nAnswer: A',
 'NLE_2015_INTRODUCTION_TO_LATIN_33': 'To answer the question, let\'s examine the text closely. In line 3, it is written: "Senātor virum comprehendit. Senātor cum virō pugnat." This translates to "The senator takes hold of the man. The senator fights with the man." Therefore, the man is fighting with the senator.\n\nAnswer: D',
 'NL

In [35]:
resp_data[q_id] = resp

In [36]:
# save
with open(resp_file, 'w') as f:
    json.dump(resp_data, f, indent=4)